# Transfer Learning: Standing on the Shoulders of Giants

**Transfer learning** is arguably the most practical technique in deep learning - used in 90% of real-world applications. Instead of training neural networks from scratch, we start with models pretrained on massive datasets and adapt them to our specific task.

## What You'll Learn

This notebook builds deep intuitions about transfer learning through hands-on implementation:

1. **What are pretrained models** and why they work so well
2. **Feature extraction** - using pretrained features as-is
3. **Fine-tuning** - adapting pretrained weights to your task
4. **When to use which approach** based on data size and domain similarity
5. **Learning rate strategies** for effective fine-tuning
6. **Practical comparisons** on CIFAR-10 using pretrained ResNet

## Why Transfer Learning Matters

Training deep neural networks from scratch requires:
- **Massive datasets** (millions of images)
- **Enormous compute** (days/weeks on GPUs)
- **Expertise** in initialization, regularization, and hyperparameter tuning

Transfer learning lets you achieve excellent results with:
- **Small datasets** (hundreds to thousands of examples)
- **Limited compute** (minutes to hours)
- **Pretrained knowledge** from models trained on ImageNet, etc.

## Setup

Let's import the necessary libraries and set up our environment.

In [ ]:
# Standard library imports

# Third-party imports

# Local library imports
    get_device, set_seed, count_parameters,
    create_dataset, create_dataloaders,
    get_dataset_config,
)

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### Set Random Seeds and Device

For reproducible results, we set random seeds. We also configure the device (GPU if available, otherwise CPU).

In [ ]:
from aiml_notebooks import (

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Configure device (prefer CPU for this notebook to avoid MPS issues with torchvision models)
device = get_device(prefer_cpu=True)
print(f"Using device: {device}")

## Part 1: Understanding Pretrained Models

### What is a Pretrained Model?

A **pretrained model** is a neural network that has already been trained on a large dataset (typically ImageNet with 1.4M images across 1000 categories). When we use a pretrained model, we get:

1. **Optimized weights** learned from millions of examples
2. **Hierarchical features** from low-level (edges, textures) to high-level (objects, scenes)
3. **Strong initializations** that converge faster and generalize better

### Why Does Transfer Learning Work?

**Key insight:** Features learned on one task are often useful for other tasks!

- **Early layers** learn universal features (edges, colors, textures)
- **Middle layers** learn patterns and parts (wheels, eyes, fur)
- **Late layers** learn task-specific combinations

These features transfer well because natural images share common structure.

### Loading a Pretrained Model

PyTorch's `torchvision.models` provides many pretrained architectures. Let's load a ResNet-18 pretrained on ImageNet.

In [ ]:
# Load pretrained ResNet-18
# weights='IMAGENET1K_V1' loads the default pretrained weights
pretrained_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("Pretrained ResNet-18 architecture:")
print(pretrained_model)
print(f"\nTotal parameters: {count_parameters(pretrained_model):,}")

### Anatomy of ResNet-18

Notice the structure:
- **conv1**: Initial 7×7 convolution (extracts low-level features)
- **layer1-4**: Residual blocks that progressively learn more complex features
- **avgpool**: Global average pooling to reduce spatial dimensions
- **fc**: Fully connected layer that outputs 1000 classes (ImageNet categories)

For transfer learning, we typically **keep conv1-layer4** (feature extractor) and **replace fc** (classifier).

### Inspecting the Classifier Head

The final layer is a linear classifier trained for ImageNet's 1000 classes. We'll need to replace this for our task.

In [ ]:
# Inspect the final fully connected layer
print("Original classifier (fc layer):")
print(pretrained_model.fc)
print(f"\nInput features: {pretrained_model.fc.in_features}")
print(f"Output classes: {pretrained_model.fc.out_features}")
print("\nThis layer outputs 1000 classes for ImageNet. We'll replace it for CIFAR-10 (10 classes).")

## Part 2: Prepare the Dataset

We'll use **CIFAR-10** as our target task. CIFAR-10 has 32×32 color images in 10 classes (airplane, car, bird, etc.).

### Why CIFAR-10?

- **Different from ImageNet**: Smaller images (32×32 vs 224×224), different classes
- **Moderate dataset size**: 50,000 training images (enough to demonstrate both approaches)
- **Fast training**: Quick experiments to compare methods

To simulate a **small dataset scenario** (more realistic for transfer learning), we'll use only a subset of the data.

### Load CIFAR-10 Dataset

We'll load CIFAR-10 using our dataset factory. For pretrained models, we need to:
1. **Resize** images from 32×32 to 224×224 (ImageNet input size)
2. **Normalize** with ImageNet mean/std (what the model was trained on)

In [ ]:
# Get CIFAR-10 configuration
dataset_config = get_dataset_config('cifar10')
class_names = dataset_config['classes']
num_classes = len(class_names)

print(f"Dataset: CIFAR-10")
print(f"Number of classes: {num_classes}")
print(f"Classes: {', '.join(class_names)}")

### Create Custom Transforms for Pretrained Models

**Important:** Pretrained models expect specific input preprocessing:
- **Image size**: 224×224 (ImageNet standard)
- **Normalization**: ImageNet mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]

We must match these exactly, or the model will perform poorly!

In [ ]:
# ImageNet normalization statistics (pretrained models expect these)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transforms: augmentation + resize + ImageNet normalization
train_transform = transforms.Compose([
    transforms.Resize(224),  # Resize to ImageNet size
    transforms.RandomHorizontalFlip(),  # Data augmentation
    transforms.RandomCrop(224, padding=16),  # Augmentation with padding
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # ImageNet normalization
])

# Test transforms: just resize + normalize (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("✓ Created transforms with ImageNet normalization")
print(f"  - Resize: 32×32 → 224×224")
print(f"  - Normalize: mean={IMAGENET_MEAN}, std={IMAGENET_STD}")

### Load Dataset with Custom Transforms

In [ ]:
# Load full CIFAR-10 dataset with custom transforms
train_dataset, test_dataset = create_dataset(
    dataset_id='cifar10',
    train_transform=train_transform,
    test_transform=test_transform
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

### Create Small Dataset Subset

To simulate a realistic transfer learning scenario (limited data), we'll use only **2,500 training examples** (5% of full dataset). This is where transfer learning really shines!

In [ ]:
from torch.utils.data import DataLoader, Subset

In [ ]:
# Create a small subset for faster experimentation
# In practice, transfer learning is most useful when you have limited data
SUBSET_SIZE = 2500
subset_indices = np.random.choice(len(train_dataset), size=SUBSET_SIZE, replace=False)
small_train_dataset = Subset(train_dataset, subset_indices)

print(f"\nUsing subset of training data:")
print(f"  Original: {len(train_dataset):,} samples")
print(f"  Subset: {len(small_train_dataset):,} samples ({len(small_train_dataset)/len(train_dataset)*100:.1f}%)")
print(f"\nThis simulates a realistic scenario where you have limited labeled data.")

### Create DataLoaders

In [ ]:
# Create data loaders
BATCH_SIZE = 32

train_loader, test_loader = create_dataloaders(
    train_dataset=small_train_dataset,
    val_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=2,
    use_collate_fn=False
)

print(f"✓ DataLoaders created (batch_size={BATCH_SIZE})")
print(f"  Train batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

## Part 3: Approach 1 - Feature Extraction

### What is Feature Extraction?

In **feature extraction** mode:
1. **Freeze all pretrained layers** (conv1 through layer4) - their weights don't change
2. **Replace the classifier head** (fc layer) with a new one for our task
3. **Train only the new classifier** using pretrained features as fixed inputs

**Analogy:** Think of the pretrained model as a high-quality feature detector. We keep this detector frozen and only train a new classifier on top.

### When to Use Feature Extraction?

Feature extraction works best when:
- ✅ **Small dataset** (hundreds to few thousand examples)
- ✅ **Similar domain** to ImageNet (natural images)
- ✅ **Limited compute** (trains much faster)
- ✅ **Risk of overfitting** (fewer trainable parameters)

### Step 1: Load Pretrained Model and Freeze Layers

We'll freeze all layers except the final classifier by setting `requires_grad=False` on their parameters.

In [ ]:
# Load pretrained ResNet-18
model_fe = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all parameters in the feature extractor
for param in model_fe.parameters():
    param.requires_grad = False

print("✓ Loaded pretrained ResNet-18")
print("✓ Froze all pretrained layers (requires_grad=False)")
print("\nFrozen parameters will not be updated during training.")

### Step 2: Replace the Classifier Head

The original `fc` layer outputs 1000 classes (ImageNet). We replace it with a new layer that outputs 10 classes (CIFAR-10).

**Important:** The new layer is randomly initialized, so `requires_grad=True` by default.

In [ ]:
# Replace the classifier head
num_features = model_fe.fc.in_features  # 512 for ResNet-18
model_fe.fc = nn.Linear(num_features, num_classes)

# Move model to device
model_fe = model_fe.to(device)

print(f"✓ Replaced classifier head")
print(f"  Old: Linear({num_features}, 1000)  [ImageNet classes]")
print(f"  New: Linear({num_features}, {num_classes})  [CIFAR-10 classes]")
print(f"\nThe new classifier is randomly initialized and trainable.")

### Step 3: Verify Which Parameters are Trainable

Let's count trainable vs frozen parameters to confirm our setup.

In [ ]:
# Count trainable and frozen parameters
trainable_params = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
frozen_params = sum(p.numel() for p in model_fe.parameters() if not p.requires_grad)
total_params = trainable_params + frozen_params

print(f"Parameter breakdown:")
print(f"  Trainable: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
print(f"  Frozen: {frozen_params:,} ({frozen_params/total_params*100:.2f}%)")
print(f"  Total: {total_params:,}")
print(f"\nWe're only training {trainable_params:,} parameters (the new classifier head)!")

### Step 4: Train the Feature Extraction Model

Since we're only training the classifier head, we can use a **higher learning rate** (the new layer needs larger updates to converge from random initialization).

In [ ]:
from tqdm.auto import tqdm

In [ ]:
# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch and return average loss and accuracy."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    """Evaluate the model and return average loss and accuracy."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(loader)
    accuracy = 100. * correct / total
    return avg_loss, accuracy

print("✓ Training and evaluation functions defined")

### Train Feature Extraction Model

We'll train for a few epochs. This should be fast since we're only updating ~5k parameters!

In [ ]:
# Training configuration for feature extraction
criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(model_fe.parameters(), lr=1e-3)  # Higher LR for new classifier
NUM_EPOCHS = 5

# Training loop
print("Training feature extraction model...\n")
history_fe = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model_fe, train_loader, criterion, optimizer_fe, device)
    
    # Evaluate
    test_loss, test_acc = evaluate(model_fe, test_loader, criterion, device)
    
    # Record history
    history_fe['train_loss'].append(train_loss)
    history_fe['train_acc'].append(train_acc)
    history_fe['test_loss'].append(test_loss)
    history_fe['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Feature extraction training complete!")
print(f"Final test accuracy: {history_fe['test_acc'][-1]:.2f}%")

### Feature Extraction Results

Notice how quickly we achieved good accuracy! With only 2,500 training examples and ~5k trainable parameters, we get competitive performance.

**Key observation:** The pretrained features are already so good that we only need to train a simple classifier on top.

## Part 4: Approach 2 - Fine-Tuning

### What is Fine-Tuning?

In **fine-tuning** mode:
1. **Start with pretrained weights** (same as feature extraction)
2. **Unfreeze some or all layers** - allow them to be updated
3. **Train with a lower learning rate** - make small adjustments to pretrained weights
4. **Often use differential learning rates** - smaller LR for early layers, larger for later layers

**Analogy:** Instead of using the feature detector as-is, we fine-tune it to better match our specific task.

### When to Use Fine-Tuning?

Fine-tuning works best when:
- ✅ **Larger dataset** (thousands to tens of thousands of examples)
- ✅ **Different domain** from ImageNet (e.g., medical images, satellite imagery)
- ✅ **More compute available** (takes longer to train)
- ✅ **Want maximum accuracy** (more parameters = more capacity)

### Step 1: Create Fine-Tuning Model

We'll create a new model and unfreeze all layers. In practice, you might:
- Unfreeze only the last few layers
- Gradually unfreeze layers during training
- Use different learning rates for different layers

In [ ]:
# Load pretrained ResNet-18 (start fresh)
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace the classifier head
num_features = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_features, num_classes)

# Move to device
model_ft = model_ft.to(device)

print("✓ Created fine-tuning model")
print("✓ All layers are trainable (requires_grad=True)")

### Step 2: Verify All Parameters are Trainable

In [ ]:
# Count trainable parameters
trainable_params_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
total_params_ft = sum(p.numel() for p in model_ft.parameters())

print(f"Parameter breakdown:")
print(f"  Trainable: {trainable_params_ft:,} ({trainable_params_ft/total_params_ft*100:.2f}%)")
print(f"  Total: {total_params_ft:,}")
print(f"\nWe're training ALL {trainable_params_ft:,} parameters!")

### Step 3: Use Lower Learning Rate for Fine-Tuning

**Critical:** When fine-tuning, use a **much lower learning rate** (typically 10-100x smaller) than training from scratch.

**Why?** Pretrained weights are already good. Large updates would destroy the learned features. We want to gently adapt them.

In [ ]:
# Training configuration for fine-tuning
# Use LOWER learning rate than feature extraction!
optimizer_ft = optim.Adam(model_ft.parameters(), lr=1e-4)  # 10x smaller than feature extraction

print("✓ Optimizer configured with lr=1e-4")
print("  This is 10x lower than feature extraction (1e-3)")
print("  Lower LR preserves pretrained features while allowing adaptation")

### Step 4: Train Fine-Tuning Model

Training will take longer since we're updating all parameters (11M vs 5k).

In [ ]:
# Training loop
print("Training fine-tuning model...\n")
history_ft = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model_ft, train_loader, criterion, optimizer_ft, device)
    
    # Evaluate
    test_loss, test_acc = evaluate(model_ft, test_loader, criterion, device)
    
    # Record history
    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['test_loss'].append(test_loss)
    history_ft['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Fine-tuning training complete!")
print(f"Final test accuracy: {history_ft['test_acc'][-1]:.2f}%")

## Part 5: Compare Feature Extraction vs Fine-Tuning

Let's visualize the training curves to compare both approaches.

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss
axes[0].plot(history_fe['train_loss'], label='Feature Extraction (train)', marker='o')
axes[0].plot(history_fe['test_loss'], label='Feature Extraction (test)', marker='o')
axes[0].plot(history_ft['train_loss'], label='Fine-Tuning (train)', marker='s')
axes[0].plot(history_ft['test_loss'], label='Fine-Tuning (test)', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
axes[1].plot(history_fe['train_acc'], label='Feature Extraction (train)', marker='o')
axes[1].plot(history_fe['test_acc'], label='Feature Extraction (test)', marker='o')
axes[1].plot(history_ft['train_acc'], label='Fine-Tuning (train)', marker='s')
axes[1].plot(history_ft['test_acc'], label='Fine-Tuning (test)', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"1. Feature Extraction final test accuracy: {history_fe['test_acc'][-1]:.2f}%")
print(f"2. Fine-Tuning final test accuracy: {history_ft['test_acc'][-1]:.2f}%")
print(f"3. Improvement from fine-tuning: {history_ft['test_acc'][-1] - history_fe['test_acc'][-1]:.2f}%")

### Analysis: Which Approach is Better?

The answer depends on your scenario:

**Feature Extraction wins when:**
- ✅ Very small dataset (risk of overfitting with fine-tuning)
- ✅ Limited compute/time
- ✅ Task is similar to ImageNet

**Fine-Tuning wins when:**
- ✅ Moderate to large dataset
- ✅ Task is different from ImageNet
- ✅ Want maximum accuracy (compute is available)

In our experiment with 2,500 examples, fine-tuning likely gives a small improvement but risks overfitting. With more data, the gap would widen.

## Part 6: Advanced Fine-Tuning - Differential Learning Rates

### The Problem with Uniform Learning Rates

When fine-tuning, using the **same learning rate for all layers** is suboptimal:

- **Early layers** (conv1, layer1): Learn universal features (edges, textures) - should change very little
- **Middle layers** (layer2, layer3): Learn intermediate features - moderate changes
- **Late layers** (layer4, fc): Learn task-specific features - larger changes needed

**Solution:** Use **differential learning rates** (also called **discriminative fine-tuning**) - different learning rates for different layer groups.

### Implementing Differential Learning Rates

PyTorch optimizers accept parameter groups with different learning rates:

In [ ]:
# Create a new model for differential LR experiment
model_diff = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model_diff.fc = nn.Linear(model_diff.fc.in_features, num_classes)
model_diff = model_diff.to(device)

# Define parameter groups with different learning rates
# Strategy: early layers get smaller LR, later layers get larger LR
param_groups = [
    {'params': model_diff.conv1.parameters(), 'lr': 1e-5},      # Very small for early conv
    {'params': model_diff.layer1.parameters(), 'lr': 1e-5},     # Small for layer1
    {'params': model_diff.layer2.parameters(), 'lr': 5e-5},     # Medium for layer2
    {'params': model_diff.layer3.parameters(), 'lr': 1e-4},     # Larger for layer3
    {'params': model_diff.layer4.parameters(), 'lr': 2e-4},     # Even larger for layer4
    {'params': model_diff.fc.parameters(), 'lr': 1e-3},         # Largest for new classifier
]

optimizer_diff = optim.Adam(param_groups)

print("✓ Created optimizer with differential learning rates:")
print("  conv1:  1e-5  (freeze early features)")
print("  layer1: 1e-5  (minimal adaptation)")
print("  layer2: 5e-5  (moderate adaptation)")
print("  layer3: 1e-4  (more adaptation)")
print("  layer4: 2e-4  (significant adaptation)")
print("  fc:     1e-3  (learn new classifier)")
print("\nThis strategy preserves early features while adapting later layers!")

### Train with Differential Learning Rates

This approach often gives the best results, combining the stability of feature extraction with the flexibility of fine-tuning.

In [ ]:
# Training loop with differential learning rates
print("Training with differential learning rates...\n")
history_diff = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model_diff, train_loader, criterion, optimizer_diff, device)
    
    # Evaluate
    test_loss, test_acc = evaluate(model_diff, test_loader, criterion, device)
    
    # Record history
    history_diff['train_loss'].append(train_loss)
    history_diff['train_acc'].append(train_acc)
    history_diff['test_loss'].append(test_loss)
    history_diff['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

print(f"\n✓ Differential LR training complete!")
print(f"Final test accuracy: {history_diff['test_acc'][-1]:.2f}%")

### Compare All Three Approaches

In [ ]:
# Final comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(history_fe['test_acc'], label='Feature Extraction', marker='o', linewidth=2)
ax.plot(history_ft['test_acc'], label='Fine-Tuning (uniform LR)', marker='s', linewidth=2)
ax.plot(history_diff['test_acc'], label='Fine-Tuning (differential LR)', marker='^', linewidth=2)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Transfer Learning Approaches Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Feature Extraction:           {history_fe['test_acc'][-1]:.2f}%")
print(f"Fine-Tuning (uniform LR):     {history_ft['test_acc'][-1]:.2f}%")
print(f"Fine-Tuning (differential LR): {history_diff['test_acc'][-1]:.2f}%")
print("="*60)

## Part 7: Decision Framework - When to Use Which Approach

Here's a practical decision matrix based on **dataset size** and **domain similarity**:

### Decision Matrix

| Dataset Size | Domain Similar to ImageNet | Domain Different from ImageNet |
|--------------|----------------------------|-------------------------------|
| **Small** (<1K) | **Feature Extraction** | **Feature Extraction** (carefully!) |
| **Medium** (1K-10K) | **Feature Extraction** or **Differential LR** | **Differential LR Fine-Tuning** |
| **Large** (>10K) | **Differential LR Fine-Tuning** | **Full Fine-Tuning** or **Train from Scratch** |

### Domain Similarity Examples

**Similar to ImageNet** (natural images):
- ✅ Pet classification (cats, dogs, etc.)
- ✅ Vehicle recognition
- ✅ Plant species identification
- ✅ Food classification

**Different from ImageNet**:
- ⚠️ Medical images (X-rays, CT scans, histopathology)
- ⚠️ Satellite imagery
- ⚠️ Microscopy images
- ⚠️ Sketch/drawing recognition
- ⚠️ Infrared/thermal images

### Learning Rate Guidelines

**Feature Extraction:**
- Classifier head: `1e-3` to `1e-2` (relatively high since it's randomly initialized)

**Uniform Fine-Tuning:**
- All layers: `1e-5` to `1e-4` (much lower than training from scratch)

**Differential Fine-Tuning:**
- Early layers: `1e-6` to `1e-5` (barely update)
- Middle layers: `1e-5` to `1e-4` (moderate updates)
- Late layers: `1e-4` to `5e-4` (larger updates)
- Classifier: `1e-3` to `1e-2` (largest updates)

**Rule of thumb:** Each layer group gets 2-5x the learning rate of the previous group.

## Part 8: Additional Fine-Tuning Strategies

### Progressive Unfreezing

Instead of unfreezing all layers at once, gradually unfreeze from top to bottom:

1. **Phase 1**: Train only classifier (feature extraction)
2. **Phase 2**: Unfreeze layer4, train with low LR
3. **Phase 3**: Unfreeze layer3, continue training
4. **Phase 4**: Unfreeze remaining layers

This approach is more stable and often gives better results.

In [ ]:
# Example of progressive unfreezing (pseudo-code, not executed)
print("Progressive Unfreezing Strategy:\n")
print("Phase 1 (epochs 1-2): Train only classifier")
print("  - Freeze: conv1, layer1-4")
print("  - Train: fc")
print("  - LR: 1e-3\n")

print("Phase 2 (epochs 3-4): Unfreeze layer4")
print("  - Freeze: conv1, layer1-3")
print("  - Train: layer4, fc")
print("  - LR: 1e-4 (layer4), 5e-4 (fc)\n")

print("Phase 3 (epochs 5-6): Unfreeze layer3")
print("  - Freeze: conv1, layer1-2")
print("  - Train: layer3-4, fc")
print("  - LR: 5e-5 (layer3), 1e-4 (layer4), 5e-4 (fc)\n")

print("Phase 4 (epochs 7+): Unfreeze all layers")
print("  - Train: all layers")
print("  - LR: differential rates for all layers")

### Warmup Strategy

When fine-tuning, the random classifier head can produce large gradients that corrupt pretrained weights. Use **warmup** to stabilize:

1. Train only the classifier for 1-2 epochs (feature extraction)
2. Then unfreeze other layers and fine-tune

This gives the classifier a chance to adapt before updating pretrained weights.

## Part 9: Common Pitfalls and Best Practices

### ❌ Common Mistakes

1. **Wrong normalization**: Using CIFAR-10 mean/std instead of ImageNet mean/std
   - Pretrained models expect ImageNet normalization!

2. **Learning rate too high**: Destroys pretrained features
   - Use 10-100x lower LR than training from scratch

3. **Learning rate too low**: Classifier doesn't converge
   - New classifier head needs higher LR than pretrained layers

4. **Wrong input size**: CIFAR-10 is 32×32, but pretrained models expect 224×224
   - Always resize to the pretrained model's expected input size

5. **Not using data augmentation**: Small datasets need augmentation
   - Standard augmentations: random crop, flip, color jitter

6. **Training too long**: Risk of overfitting with small datasets
   - Use early stopping or validation monitoring

### ✅ Best Practices

1. **Always start with feature extraction** as a baseline
   - Fast, low risk of overfitting
   - If results are good enough, you're done!

2. **Match preprocessing to pretrained model**
   - Use ImageNet mean/std for ImageNet-pretrained models
   - Resize images to expected input size (usually 224×224)

3. **Use differential learning rates** when fine-tuning
   - Early layers: very small LR
   - Late layers: larger LR
   - Classifier: largest LR

4. **Monitor both train and val metrics**
   - Large train-val gap = overfitting
   - Both poor = need more capacity or better data

5. **Use learning rate scheduling**
   - Reduce LR on plateau
   - Cosine annealing
   - Warmup for first few epochs

6. **Try multiple pretrained architectures**
   - ResNet: Good all-around choice
   - EfficientNet: Best accuracy/compute tradeoff
   - Vision Transformer: State-of-art but needs more data

## Key Takeaways

### Core Concepts

1. **Transfer learning** lets you leverage knowledge from massive datasets (ImageNet) for your specific task

2. **Feature extraction** freezes pretrained layers and trains only the classifier
   - Best for: small datasets, similar domains, limited compute
   - Pros: fast, stable, low risk of overfitting
   - Cons: less flexibility, may not fully adapt to new domain

3. **Fine-tuning** updates pretrained weights with small learning rates
   - Best for: larger datasets, different domains, max accuracy
   - Pros: more flexible, can achieve higher accuracy
   - Cons: slower, risks overfitting or corrupting features

4. **Differential learning rates** are the sweet spot
   - Early layers: barely change (universal features)
   - Late layers: adapt more (task-specific features)
   - Classifier: change most (completely new task)

5. **Preprocessing matters**: Always match the pretrained model's expected inputs
   - Image size (usually 224×224)
   - Normalization (ImageNet mean/std)

### Decision Rules

**Start simple:**
1. Try feature extraction first (always!)
2. If not good enough, try fine-tuning with differential LR
3. If still not good enough, try progressive unfreezing
4. Only train from scratch if you have 10K+ examples and pretrained models don't help

**Learning rates:**
- Feature extraction classifier: `1e-3`
- Fine-tuning (uniform): `1e-4` to `1e-5`
- Fine-tuning (differential): `1e-5` (early) to `1e-3` (classifier)

### Why Transfer Learning is Everywhere

90% of real-world computer vision applications use transfer learning because:
- ✅ Most tasks have small datasets (<10K images)
- ✅ Training from scratch requires massive compute
- ✅ Pretrained features generalize surprisingly well
- ✅ Fast iteration: experiment and deploy quickly

**Remember:** You don't need millions of images or weeks of training. With just hundreds of examples and a few hours, transfer learning can achieve state-of-the-art results!

## Further Exploration

Try these experiments to deepen your understanding:

1. **Vary dataset size**: Try 100, 500, 1000, 5000 examples
   - How does the feature extraction vs fine-tuning gap change?

2. **Try different architectures**: ResNet-50, EfficientNet-B0, MobileNetV3
   - Do larger models always win with small datasets?

3. **Experiment with learning rates**: Try 10x higher and 10x lower
   - What happens to training stability and final accuracy?

4. **Progressive unfreezing**: Implement the phased approach
   - Does it improve over simple fine-tuning?

5. **Different datasets**: Try Fashion-MNIST, SVHN, or your own data
   - How does domain similarity affect transfer learning effectiveness?

6. **Learning rate schedules**: Add cosine annealing or reduce-on-plateau
   - Can you squeeze out extra performance?

Happy experimenting! 🚀